# 00 · Exploratory data analysis

Facts about the provided data that drive every design decision. Nothing here writes artifacts.

| Finding | Consequence |
|---|---|
| 0 of 7.6M matched S2/S3 records belongs to two S1 entities | hard **exclusivity** at decision time + target-competition features |
| 0 true pairs cross countries | candidate generation runs **per country** |
| ~5.6% of S1 are singletons (both countries) | empty prediction must be a first-class decision |
| S2/S3 are not deduplicated (avg 3.46 matches / S1, max 11) | true matches are noisy copies of each other → sibling-similarity features |
| names are heavily corrupted, addresses mostly intact | address-aware retrieval channels; name-only retrieval is weak for India |
| Indic scripts (Devanagari, Telugu, …) in S2/S3 names and states | transliteration + phonetic skeleton |
| test adds **France** (unseen) | country is never a model feature; language-agnostic features only |

In [ ]:
# --- Setup: make src/ importable, load run settings -------------------------
import os, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "entity_forge").is_dir())
sys.path.insert(0, str(ROOT / "src"))

# Override settings here or with EF_* environment variables before starting Jupyter.
# os.environ["EF_DEV_MODE"] = "1"     # small consistent slice: end-to-end smoke test on a laptop
# os.environ["EF_N_THREADS"] = "32"

import polars as pl
from entity_forge import stages
from entity_forge.settings import Settings

pl.Config.set_tbl_rows(30); pl.Config.set_fmt_str_lengths(80); pl.Config.set_tbl_width_chars(220)
stages.setup_logging()
S = Settings.from_env()
print(f"root={S.root}\nwork_dir={S.work_dir}\ndev_mode={S.dev_mode} threads={S.n_threads}")

In [ ]:
from entity_forge.io import read_source, read_ground_truth_pairs

gt = read_ground_truth_pairs(S.ground_truth)
s1 = read_source(S.raw("train", 1))
targets = pl.concat([read_source(S.raw("train", i)) for i in (2, 3)])
print("train S1", s1.height, "| S2+S3", targets.height, "| true pairs", gt.height)

## Match-count distribution and singletons

In [ ]:
per_s1 = pl.DataFrame({"s1_id": s1["entity_id"]}).join(gt.group_by("s1_id").len("n"), on="s1_id", how="left").fill_null(0)
display(per_s1.group_by("n").len().sort("n"))
display(per_s1.join(s1.select(pl.col("entity_id").alias("s1_id"), "country"), on="s1_id")
        .group_by("country").agg((pl.col("n") == 0).mean().alias("singleton_rate"), pl.col("n").mean().alias("avg_matches")))

## Exclusivity: can one S2/S3 record match two S1 entities?

In [ ]:
shared = gt.group_by("t_id").len().filter(pl.col("len") > 1).height
print(f"S2/S3 ids under more than one S1: {shared} of {gt['t_id'].n_unique()}")

## Do true pairs ever cross countries?

In [ ]:
cc = (gt.join(s1.select(pl.col("entity_id").alias("s1_id"), pl.col("country").alias("c1")), on="s1_id")
        .join(targets.select(pl.col("entity_id").alias("t_id"), pl.col("country").alias("c2")), on="t_id"))
print("cross-country true pairs:", cc.filter(pl.col("c1") != pl.col("c2")).height)

## Noise patterns: an S1 record next to its true matches

In [ ]:
sample = gt.filter(pl.col("s1_id").is_in(gt["s1_id"].unique().sample(6, seed=7).implode()))
view = (sample.join(s1.select(pl.col("entity_id").alias("s1_id"), pl.col("business_name").alias("s1_name"),
                               pl.col("business_address").alias("s1_addr")), on="s1_id")
        .join(targets.select(pl.col("entity_id").alias("t_id"), pl.col("business_name").alias("t_name"),
                              pl.col("business_address").alias("t_addr")), on="t_id")
        .sort("s1_id"))
view

## Scripts and missing fields

In [ ]:
def profile(df, label):
    return df.select(
        pl.lit(label).alias("source"),
        pl.col("business_name").str.contains(r"[\u0900-\u0DFF]").mean().alias("indic_name"),
        pl.col("business_name").str.contains(r"[^\x00-\x7F]").mean().alias("non_ascii_name"),
        pl.col("business_address").is_null().mean().alias("missing_address"),
    )
pl.concat([profile(s1, "train S1"), profile(targets, "train S2+S3"),
           profile(read_source(S.raw("test", 1)), "test S1")])

## Test countries (France is new)

In [ ]:
pl.concat([read_source(S.raw("test", i)).group_by("country").len().with_columns(pl.lit(f"test S{i}").alias("source"))
           for i in (1, 2, 3)]).pivot(on="source", index="country", values="len")